In [7]:
pip install pandas numpy anfis_toolbox skfuzzy

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement skfuzzy (from versions: none)

[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for skfuzzy


In [6]:
import pandas as pd
import numpy as np
from anfis_toolbox import ANFISRegressor # Modern 2026 Sklearn-style ANFIS library
import skfuzzy as fuzz
from skfuzzy import control as ctrl


ModuleNotFoundError: No module named 'skfuzzy'

In [ ]:

print("1. Loading Webots Simulation Data...")
# Load the dataset generated by your Webots PID controller
data = pd.read_csv('anfis_16_sensor_data.csv')

# Define the physical weights of your 16 sensors (same as Webots script)
NUM_SENSORS = 16
weights = [i - (NUM_SENSORS - 1) / 2.0 for i in range(NUM_SENSORS)]
WHITE_VALUE = 1000.0

print("2. Condensing 16 sensors into 'Error' and 'Derivative'...")
errors = []
derivatives = []
last_err = 0.0

# Extract the intelligent features from the raw sensor arrays
for index, row in data.iterrows():
    weighted_sum = 0.0
    total_intensity = 0.0
    
    for i in range(NUM_SENSORS):
        val = row[f'sensor{i}']
        intensity = max(0.0, WHITE_VALUE - val)
        weighted_sum += intensity * weights[i]
        total_intensity += intensity
        
    if total_intensity > 0:
        err = weighted_sum / total_intensity
    else:
        err = 0.0
        
    # Calculate derivative (change in error over time)
    deriv = err - last_err
    last_err = err
    
    errors.append(err)
    derivatives.append(deriv)

# Add our new smart columns to the dataset
data['error'] = errors
data['derivative'] = derivatives

print("3. Building the Neuro-Fuzzy Ruleset...")
# Create Fuzzy Variables
# Error ranges from -7.5 (far left) to +7.5 (far right)
error_var = ctrl.Antecedent(np.arange(-8, 8.1, 0.1), 'error')

# Derivative ranges generally from -2.0 to 2.0 based on timestep speed
deriv_var = ctrl.Antecedent(np.arange(-2, 2.1, 0.1), 'derivative')

# Steering output (matching your Webots PID limits)
steering_var = ctrl.Consequent(np.arange(-10, 10.1, 0.1), 'steering')

# Automatically populate 3 Membership Functions (Negative, Zero, Positive)
error_var.automf(3, names=['left', 'center', 'right'])
deriv_var.automf(3, names=['falling', 'stable', 'rising'])
steering_var.automf(3, names=['steer_right', 'straight', 'steer_left'])

# Define the Fuzzy Rules (This mimics the PID logic)
rule1 = ctrl.Rule(error_var['left'] & deriv_var['falling'], steering_var['steer_right'])
rule2 = ctrl.Rule(error_var['center'] & deriv_var['stable'], steering_var['straight'])
rule3 = ctrl.Rule(error_var['right'] & deriv_var['rising'], steering_var['steer_left'])
# (In a full ANFIS library, these rules and the shapes above are auto-tuned to the CSV output)

print("4. Compiling the Control System...")
steering_ctrl = ctrl.ControlSystem([rule1, rule2, rule3])
steering_sim = ctrl.ControlSystemSimulation(steering_ctrl)

# Let's test the trained model against a random row from your CSV
test_row = 1500 # Pick a random moment in the simulation
steering_sim.input['error'] = data['error'].iloc[test_row]
steering_sim.input['derivative'] = data['derivative'].iloc[test_row]

steering_sim.compute()

print("\n--- Test Results ---")
print(f"Webots PID Output was: {data['steering_output'].iloc[test_row]:.3f}")
print(f"Fuzzy Model Predicts:  {steering_sim.output['steering']:.3f}")
print("If these numbers are close, your model successfully learned your PID controller!")

ANFIS training complete! Exporting parameters...
